# Rendering MuJoCo Simulations with Mediapy

This notebook demonstrates how to simulate a MuJoCo model (`centipede.xml`) and render the output as a video directly in the notebook using `mediapy`. This is the standard alternative to using `mujoco.viewer` when running in headless environments or when you need to save simulation footage.

### 1. Imports
We import the physics bindings (`mujoco`) and the visualization library (`mediapy`).

In [1]:
import mujoco
import mediapy as media
import numpy as np


### 2. Load the Physics Model
Here we load the XML file. 
* **`model`**: The static blueprint (lengths, masses, shapes, mesh geometry).
* **`data`**: The dynamic state (positions, velocities, forces, contact points).

*Note: Ensure `centipede.xml` is in the current directory.*

In [2]:
xml_path = 'centipede.xml'

# Load the model and data
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)


### 3. Initialize the Renderer
The renderer acts as the camera. It converts the 3D mathematical model into 2D pixels based on the current state of `data`.

In [3]:
renderer = mujoco.Renderer(model)


### 4. The Simulation Loop
This is the core logic. We must synchronize the physics engine with the video framerate.

1.  **Calculate Steps:** MuJoCo runs at a very high frequency (e.g., 500Hz, or 0.002s timestep). Video runs at a low frequency (e.g., 30Hz). We calculate `physics_steps_per_frame` to determine how many physics calculations correspond to one frame of video.
2.  **Step Physics:** Advance the world state using `mujoco.mj_step`.
3.  **Render:** Capture the pixels using `renderer.render()`.

In [4]:
# Simulation parameters
duration = 3.0  # (seconds)
framerate = 30  # (Hz)
frames = []

# Calculate how many physics steps fit into one video frame
# Formula: (1 / fps) / timestep
physics_steps_per_frame = int(1.0 / framerate / model.opt.timestep)

print(f"Simulating {duration} seconds...")

# Reset data just in case we are re-running this cell
mujoco.mj_resetData(model, data)

for i in range(int(duration * framerate)):
    # 1. Advance physics multiple times (filling the gap between video frames)
    for _ in range(physics_steps_per_frame):
        mujoco.mj_step(model, data)
    
    # 2. Update the visual scene with the new positions
    renderer.update_scene(data)
    
    # 3. Capture pixels and store them
    pixels = renderer.render()
    frames.append(pixels)

print(f"Simulation complete. Captured {len(frames)} frames.")


Simulating 3.0 seconds...
Simulation complete. Captured 90 frames.


### 5. Display Video
Finally, we use `mediapy` to compile the list of frames into a video widget displayed inline.

In [5]:
media.show_video(frames, fps=framerate)


This browser does not support the video tag.
